# Experiment 4（前段）: ベースモデルの Known / Unknown プローブ

**やること**: 評価 QA（`make_qa.py` の出力）を、fine-tuning 前のベースモデルに closed-book（文脈なし）で解かせる。
**英語でも日本語でも一度も正解できなかった事実**だけを注入対象にする。

**なぜ必要か**: ベースモデルがすでに答えられる事実を注入すると、fine-tuning 後に日本語で正解しても
「英語から転移した」のか「もともと知っていた」のか区別できない。

**判定ルール**（Gekhman et al. 2024 の *Unknown* の定義と同じ）

- 1 事実・1 言語につき、few-shot プロンプト `N_EX` 種類 × 質問の言い回し 2 本 = `2·N_EX` 個のプロンプトを作る
- 各プロンプトで greedy 1 回 ＋ サンプリング `N_SAMPLES` 回（T=0.5）答えさせる
- **1 回でも正解 → `Known`、1 回も正解しない → `Unknown`**

1 回の greedy だけで判定しない理由: 「たまにしか答えられない」事実（例: 日本の首都を greedy では簡体字「东京」と答え、サンプルで「東京」と答える）が
Unknown に紛れ込み、注入後の正解が転移に見えてしまうため。何度も聞いて一度も当たらないものだけを Unknown にする。

（論文はさらに Known を HighlyKnown / MaybeKnown / WeaklyKnown に分けるが、この研究は Unknown しか使わないので分けない。）

**入力**: `experiment_data/eval_qa.jsonl`, `experiment_data/facts.jsonl`
**出力**: `experiment_data/facts_probed.jsonl`（事実行 + `known_en` / `known_ja` + 生の回答）、`experiment_data/probe_summary.json`

**実行**: 上から順に実行する。まず `LIMIT = 20` で試し、問題なければ `LIMIT = 0`（全件）。
生成結果は `CHUNK` 事実ごとに `probe_cache/` に保存するので、**途中で止まっても同じセルを再実行すれば続きから**。

In [ ]:
# ===== Config =====  papermill の "parameters" セル
model_name = "Qwen/Qwen2.5-15B"   # fine-tuning に使うベースモデルと同じもの

QA_FILE = "experiment_data/eval_qa.jsonl"
FACTS_FILE = "experiment_data/facts.jsonl"
OUT_FILE = "experiment_data/facts_probed.jsonl"
SUMMARY_FILE = "experiment_data/probe_summary.json"
CACHE_DIR = "experiment_data/probe_cache"

N_EX = 4           # few-shot プロンプトの種類数（最大 4）
N_SAMPLES = 8      # プロンプトごとのサンプリング回数
TEMPERATURE = 0.5
BATCH = 32         # GPU メモリに余裕があれば 64
CHUNK = 200        # この事実数ごとにキャッシュへ保存（中断しても失うのは最大この分だけ）
LIMIT = 0          # 0 = 全事実。試し: 20
RUN_CONTROL = True # 既知 / 架空の事実で判定が正しく動くかを先に確認する

In [ ]:
import glob, hashlib, json, os, shutil, sys
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
# プロンプト（FEWSHOT_SETS）・正誤判定（is_correct）・正解表記（golds_for）・コントロールセットだけ借りる
from probe_known import CONTROL, FEWSHOT_SETS, golds_for, is_abstain, is_correct

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# 新しい PyTorch は CUDA 上で Triton カーネルを使い、Triton は初回に C コンパイラで小さなモジュールをビルドする。
# コンパイラが無いと生成の途中で "Failed to find C compiler" で落ちるので、ここで先に確認する。
if device == "cuda" and not os.environ.get("CC"):
    cc = (shutil.which("gcc") or shutil.which("cc") or shutil.which("clang")
          or next(iter(glob.glob(os.path.join(sys.prefix, "bin", "*-linux-gnu-gcc"))), None))
    if cc is None:
        raise RuntimeError("C コンパイラが見つからない。次のどれかをしてからカーネルを再起動する:\n"
                           "  (1) ノートブックで  !conda install -y -c conda-forge gcc\n"
                           "  (2) ターミナルで    module load gcc   （クラスタに module がある場合）\n"
                           "  (3) gcc の場所が分かっているなら  os.environ['CC'] = '/path/to/gcc'")
    os.environ["CC"] = cc
    print("C compiler:", cc)

model_slug = model_name.split("/")[-1]
print(f"device {device} | {N_EX} few-shot × 2 phrasings × (1 greedy + {N_SAMPLES} samples) "
      f"= {2 * N_EX * (1 + N_SAMPLES)} answers per fact per language")

## 1. モデル読み込みと判定関数

In [ ]:
tok = AutoTokenizer.from_pretrained(model_name)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
dtype = torch.float16 if device != "cpu" else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device).eval()
NEWLINE = tok("\n", add_special_tokens=False).input_ids[-1]   # 改行で生成を止める（答えは 1 行）


@torch.no_grad()
def generate(prompts, n_samples=0):
    """n_samples=0 なら greedy 1 回、それ以外はサンプリング n_samples 回。プロンプトごとに回答のリストを返す。"""
    n = max(1, n_samples)
    step = max(1, BATCH // n)
    out = []
    for i in range(0, len(prompts), step):
        enc = tok(prompts[i:i + step], return_tensors="pt", padding=True).to(device)
        if n_samples:
            kw = dict(do_sample=True, temperature=TEMPERATURE, top_p=1.0, num_return_sequences=n)
        else:
            kw = dict(do_sample=False)
        ids = model.generate(**enc, max_new_tokens=20, pad_token_id=tok.pad_token_id,
                             eos_token_id=[tok.eos_token_id, NEWLINE], **kw)
        texts = tok.batch_decode(ids[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        texts = [t.split("\n")[0].strip() for t in texts]
        out += [texts[j:j + n] for j in range(0, len(texts), n)]
    return out


def probe(questions_per_fact, golds_per_fact, lang):
    """事実ごとに Known / Unknown を判定する。
    questions_per_fact: 事実ごとの質問文リスト（評価 QA は 2 本、コントロールは 1 本）
    golds_per_fact:     事実ごとの正解表記リスト（answer + aliases）"""
    prompts, owner = [], []
    for i, questions in enumerate(questions_per_fact):
        for q in questions:
            for template in FEWSHOT_SETS[lang][:N_EX]:
                prompts.append(template.format(q=q))
                owner.append(i)

    greedy = generate(prompts)
    samples = generate(prompts, N_SAMPLES) if N_SAMPLES else [[] for _ in prompts]

    rows = [{"greedy": [], "samples": []} for _ in questions_per_fact]
    for i, g, s in zip(owner, greedy, samples):
        rows[i]["greedy"] += g
        rows[i]["samples"] += s
    for r, golds in zip(rows, golds_per_fact):
        answers = r["greedy"] + r["samples"]
        r["n_correct"] = sum(is_correct(a, golds) for a in answers)
        r["n_total"] = len(answers)
        r["label"] = "Known" if r["n_correct"] > 0 else "Unknown"
        r["abstain_greedy"] = sum(is_abstain(a, lang) for a in r["greedy"])   # 「わかりません」と答えた回数
    return rows

## 2. コントロールセット（判定が正しく動くかの確認）

常識的な事実（日本の首都など）は `Known`、架空の企業とカットオフ後の事実（Codex CLI）は `Unknown` になるはず。
英語で常識的な事実が `Unknown` になるなら、プロンプトか判定に問題があるので本番に進まない。
（日本語は、ベースモデルが日本語で答えられず `Unknown` になるものがある。これは判定の問題ではなくモデルの限界。）

In [ ]:
if RUN_CONTROL:
    ctrl = {"en": probe([[c[0]] for c in CONTROL], [c[2] for c in CONTROL], "en"),
            "ja": probe([[c[1]] for c in CONTROL], [c[3] for c in CONTROL], "ja")}
    display(pd.DataFrame([{
        "question": c[0][:45], "expected": c[4],
        "en": ctrl["en"][i]["label"], "en_correct": f'{ctrl["en"][i]["n_correct"]}/{ctrl["en"][i]["n_total"]}',
        "en_answer": ctrl["en"][i]["greedy"][0],
        "ja": ctrl["ja"][i]["label"], "ja_correct": f'{ctrl["ja"][i]["n_correct"]}/{ctrl["ja"][i]["n_total"]}',
        "ja_answer": ctrl["ja"][i]["greedy"][0],
    } for i, c in enumerate(CONTROL)]))

## 3. プローブ本体

`CHUNK` 事実ずつ処理し、終わるたびに `probe_cache/` に保存する。再実行すると保存済みの分は読み込むだけ。
進捗バーの残り時間で全体の所要時間がわかる。

In [ ]:
facts = {}
with open(FACTS_FILE, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        facts[row["fact_id"]] = row
with open(QA_FILE, encoding="utf-8") as f:
    qa = [json.loads(line) for line in f]
qa = [q for q in qa if q["fact_id"] in facts]
if LIMIT:
    qa = qa[:LIMIT]
print(f"probing {len(qa)} facts")

cache_dir = Path(CACHE_DIR) / f"{model_slug}-ex{N_EX}-s{N_SAMPLES}-t{TEMPERATURE}"
cache_dir.mkdir(parents=True, exist_ok=True)
results = {q["fact_id"]: {} for q in qa}

for lang in ("en", "ja"):
    for start in tqdm(range(0, len(qa), CHUNK), desc=lang):
        part = qa[start:start + CHUNK]
        questions = [q[f"q_{lang}"][:2] for q in part]
        golds = [golds_for(q, lang) for q in part]
        # 質問・正解が変わったら別ファイルになる（古いキャッシュを誤って使わない）
        key = hashlib.md5(json.dumps([[q["fact_id"] for q in part], questions, golds],
                                     ensure_ascii=False).encode()).hexdigest()[:10]
        path = cache_dir / f"{lang}-{start:05d}-{key}.json"
        if path.exists():
            rows = json.loads(path.read_text(encoding="utf-8"))
        else:
            rows = probe(questions, golds, lang)
            tmp = path.with_suffix(".tmp")
            tmp.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
            tmp.rename(path)   # 書き込み途中で止まっても壊れたキャッシュが残らない
        for q, r in zip(part, rows):
            results[q["fact_id"]][lang] = r

## 4. 結果

- **両言語 Unknown** の数 = 注入対象（目安: 300 を下回ったら記事を追加収集）
- EN × JA のクロス表
- greedy での棄権率（「わかりません」と答える割合）の言語差。言語によって棄権しやすさが違うと評価の交絡になるので、ベースラインとして記録する

In [ ]:
df = pd.DataFrame([{
    "fact_id": q["fact_id"], "article": q["article"], "type": q["type"],
    "known_en": results[q["fact_id"]]["en"]["label"], "known_ja": results[q["fact_id"]]["ja"]["label"],
    "correct_en": results[q["fact_id"]]["en"]["n_correct"], "correct_ja": results[q["fact_id"]]["ja"]["n_correct"],
    "abstain_en": results[q["fact_id"]]["en"]["abstain_greedy"], "abstain_ja": results[q["fact_id"]]["ja"]["abstain_greedy"],
} for q in qa])
df["inject"] = (df["known_en"] == "Unknown") & (df["known_ja"] == "Unknown")

n_greedy = 2 * N_EX
cross = pd.crosstab(df["known_en"], df["known_ja"], margins=True)
abstain = {lang: round(df[f"abstain_{lang}"].sum() / (n_greedy * len(df)), 3) for lang in ("en", "ja")}
by_type = df.groupby("type")["inject"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "facts", "sum": "inject", "mean": "inject_rate"}).round(3)

print(f"注入対象（両言語 Unknown）: {int(df['inject'].sum())} / {len(df)} 事実")
print("\nEN（行）× JA（列）:\n" + cross.to_string())
print(f"\ngreedy の棄権率: en {abstain['en']:.1%}   ja {abstain['ja']:.1%}")
print("\n事実の型ごとの注入対象:\n" + by_type.to_string())

In [ ]:
def show(fact_id):
    """1 事実の質問・正解・全回答を表示する（✓ = 正解判定）。"""
    q = next(x for x in qa if x["fact_id"] == fact_id)
    print(f"=== {fact_id}  [{q['type']}]  {q['article']}   正解: {q['answer']!r}")
    for lang in ("en", "ja"):
        r = results[fact_id][lang]
        print(f"\n  [{lang}] {r['label']}  正解 {r['n_correct']}/{r['n_total']}")
        for question in q[f"q_{lang}"][:2]:
            print(f"    Q: {question}")
        for a in r["greedy"]:
            print(f"    greedy {'✓' if is_correct(a, golds_for(q, lang)) else ' '} {a!r}")
        print(f"    samples（先頭 8 件）: {r['samples'][:8]}")


# 例: Known になった事実と Unknown になった事実を 1 件ずつ見る
for label in ("Known", "Unknown"):
    hit = df[df["known_en"] == label]
    if len(hit):
        show(hit.iloc[0]["fact_id"])
        print()

## 5. 保存

In [ ]:
with open(OUT_FILE, "w", encoding="utf-8") as out:
    for q in qa:
        row = dict(facts[q["fact_id"]])
        r = results[q["fact_id"]]
        row["known_en"], row["known_ja"] = r["en"]["label"], r["ja"]["label"]
        row["probe"] = r   # 言語ごとの {label, n_correct, n_total, greedy, samples, abstain_greedy}
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "model": model_name, "n_ex": N_EX, "n_samples": N_SAMPLES, "temperature": TEMPERATURE,
    "n_facts": len(df), "n_inject": int(df["inject"].sum()),
    "known": {lang: df[f"known_{lang}"].value_counts().to_dict() for lang in ("en", "ja")},
    "cross_en_ja": pd.crosstab(df["known_en"], df["known_ja"]).to_dict(),
    "abstain_rate_greedy": abstain,
    "by_type": by_type.to_dict("index"),
}
with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=1)
print(f"wrote {len(qa)} facts → {OUT_FILE}\nsummary → {SUMMARY_FILE}")

## 6. 次のステップ

```bash
# 注入対象（両言語 Unknown）を含む記事だけ言い換えを作る（OpenAI API。ローカルで可）
python scripts/make_paraphrases.py --facts experiment_data/facts_probed.jsonl --unknown-only --n 6 --model gpt-5-mini
```